# Stopword & noise-phrase list maintenance

Keeps the `public.stopwords` table up to date by mining
`public.ngrams_summary` for words and phrases (`n_gram` 1-4) that occur
almost every day across most sources — a strong signal of a
generic/function word or boilerplate phrase rather than a real trend.

The list is size-independent: single words and multi-word phrases live in
the same table and are matched against as one list in golang, regardless of
how many words an entry has.

Entries are never rewritten or removed - each run only *inserts* rows that
aren't already present (anything previously mined is treated as "known" on
the next run and won't be re-proposed). Query the `public.stopwords` table
to review or delete a run's additions.

**Setup (once):**
```bash
cd notebooks
uv sync
```


In [ ]:
import os
import re
from pathlib import Path

import pandas as pd
import psycopg
from dotenv import load_dotenv

load_dotenv(Path('..') / '.env')

conninfo = psycopg.conninfo.make_conninfo(
    host=os.environ['POSTGRES_HOST'],
    port=int(os.getenv('POSTGRES_PORT', '5432')),
    user=os.environ['POSTGRES_USER'],
    password=os.environ['POSTGRES_PASSWORD'],
    dbname=os.environ['POSTGRES_DATABASE'],
    sslmode=os.getenv('PGSSLMODE', 'require'),
)


def q(sql: str, params: dict | None = None) -> pd.DataFrame:
    with psycopg.connect(conninfo) as conn, conn.cursor() as cur:
        cur.execute(sql, params or None)
        cols = [d.name for d in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=cols)


STOPWORDS_TABLE = 'public.stopwords'
print('Ready')
print('Stopwords table:', STOPWORDS_TABLE)


## Parameters

- `languages`: which `ngrams_summary.language` values to mine.
- `window_days`: how far back to look for the ubiquity calculation.
- `min_day_ubiquity` / `min_total_freq`: thresholds for single words.
- `min_day_ubiquity_phrase` / `min_total_freq_phrase`: separate, stricter
  thresholds for multi-word phrases (`n_gram` 2-4) - phrase space is much
  bigger, so genuinely constant boilerplate is rarer and needs less evidence
  to trust than a single word does.
- `ALLOWLIST`: domain/topical words or phrases (any size) that must never be
  auto-added, even if they are statistically ubiquitous. Extend this over
  time.


In [ ]:
params = {
    'languages': ['en', 'de'],
    'window_days': 90,
    'min_day_ubiquity': 0.75,
    'min_total_freq': 300,
    'min_day_ubiquity_phrase': 0.6,
    'min_total_freq_phrase': 100,
}


def load_known() -> set[str]:
    known_df = q(f'SELECT stopword FROM {STOPWORDS_TABLE}')
    if known_df.empty:
        return set()
    return set(known_df['stopword'].str.lower())


def append_new_entries(new_entries: list[tuple[str, bool]]) -> None:
    if not new_entries:
        print('No new candidates - nothing to insert.')
        return

    # Normalize and deduplicate by stopword while preserving the first chosen exact flag.
    cleaned_pairs = []
    seen_words = set()
    for stopword, exact in new_entries:
        word = (stopword or '').strip().lower()
        if not word or word in seen_words:
            continue
        seen_words.add(word)
        cleaned_pairs.append((word, bool(exact)))

    if not cleaned_pairs:
        print('No new candidates - nothing to insert.')
        return

    inserted_total = 0
    for exact_value in (True, False):
        words = sorted([word for word, exact in cleaned_pairs if exact == exact_value])
        if not words:
            continue

        inserted = q(
            f'INSERT INTO {STOPWORDS_TABLE} (stopword, exact) '
            'SELECT unnest(%(words)s::text[]), %(exact)s '
            'ON CONFLICT DO NOTHING '
            'RETURNING stopword',
            {'words': words, 'exact': exact_value},
        )
        inserted_total += len(inserted)

    print(f"Inserted {inserted_total} new entrie(s) into {STOPWORDS_TABLE}")

## Mine `ngrams_summary` for ubiquitous multi-word phrases

Same idea as the single-word mining above, but for `n_gram` 2-4: phrases
that appear on almost every day, across most sources, regardless of what's
actually in the news that day. Real trends are bursty by definition, so
near-constant phrases are boilerplate/navigation noise ("read more", "sign
up for", "cookie policy", ...), not trends - and get inserted into the same
`stopwords` table as the single words above.

Already-known noise (HTML/wiki/parser/style artifacts) is excluded up front
using the same regexes as [`ngrams.go`](../golang/internal/ngrams/ngrams.go)
and `trending.sql`, so this surfaces genuinely *new* noise phrases.


In [ ]:
mining_sql_phrases = '''
WITH bounds AS (
    SELECT language, max(day) AS ref_day
    FROM public.ngrams_summary
    WHERE n_gram BETWEEN 1 AND 4
    GROUP BY language
),
recent AS (
    SELECT s.words, s.n_gram, s.language, s.day, s.sources, s.frequencies
    FROM public.ngrams_summary s
    JOIN bounds b USING (language)
    WHERE s.n_gram BETWEEN 2 AND 4
      AND s.language = ANY(%(languages)s)
      AND s.day > b.ref_day - %(window_days)s
      AND s.words ~ '[[:alpha:]]'
      AND s.words !~ '^[0-9]+([[:space:]]+[0-9]+)*$'
      AND lower(s.words) !~ '(^|[[:space:]])(39|34|gt|lt)([[:space:]]|$)'
),
window_days AS (
    SELECT language, count(DISTINCT day) AS total_days
    FROM recent
    GROUP BY language
)
SELECT
    r.words,
    r.n_gram,
    r.language,
    count(DISTINCT r.day)                                   AS days_seen,
    sum(r.sources)                                          AS source_hits,
    sum(r.frequencies)                                      AS total_freq,
    w.total_days,
    round(count(DISTINCT r.day)::numeric / w.total_days, 3) AS day_ubiquity
FROM recent r
JOIN window_days w USING (language)
GROUP BY r.words, r.n_gram, r.language, w.total_days
HAVING sum(r.frequencies) >= %(min_total_freq_phrase)s
ORDER BY day_ubiquity DESC, total_freq DESC
'''

mined_phrases = q(mining_sql_phrases, params)
print(f"{len(mined_phrases)} phrases pass the min_total_freq_phrase threshold")
mined_phrases.head(20)

## Filter down to genuinely new phrase candidates

Apply `min_day_ubiquity_phrase`, drop anything already listed in the
`stopwords` table (reloaded to pick up words inserted above), and drop
anything on `ALLOWLIST`. What's left is shown for review below - these are
the phrases that would be inserted into the table.


In [ ]:
known = load_known()

eligible_phrases = mined_phrases[mined_phrases['day_ubiquity'] >= params['min_day_ubiquity_phrase']]
candidate_phrases = (
    eligible_phrases[
        ~eligible_phrases['words'].isin(known)
    ]
    .sort_values(['day_ubiquity', 'total_freq'], ascending=False)
    .reset_index(drop=True)
)

print(f"{len(known)} entries already known, {len(eligible_phrases)} pass ubiquity threshold")
print(f"{len(candidate_phrases)} new candidate(s) - review before trusting blindly")
candidate_phrases
